In [0]:
%sql
-- How has permit volume trended month over month and year over year?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_permit_volume AS
WITH monthly_data AS (
    SELECT 
    dd.year,
    dd.month,
    COUNT(fp.permit_nbr) AS permit_count
    FROM la_lakehouse.gold.fact_permits AS fp
    LEFT JOIN la_lakehouse.gold.dim_date AS dd
    ON fp.submitted_date_key = dd.date_key
    WHERE dd.date_key IS NOT NULL
    GROUP BY dd.year, dd.month
)
SELECT 
month, 
year, 
permit_count,
LAG(permit_count, 1) OVER (ORDER BY year, month) AS prev_month_count,
ROUND(
    (permit_count - LAG(permit_count,1) OVER (ORDER BY year, month)) * 100.0 / LAG(permit_count,1) OVER (ORDER BY year, month),2
) AS mom_growth_pct, 
LAG(permit_count, 12) OVER (ORDER BY year, month) AS prev_year_count,
ROUND(
    (permit_count - LAG(permit_count,12) OVER (ORDER BY year, month)) * 100.0 / LAG(permit_count,12) OVER (ORDER BY year, month),2
) AS yoy_growth_pct
FROM monthly_data
ORDER BY year, month

In [0]:
%sql 
-- Which permit type/sub-type combinations are growing fastest vs. declining?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_permit_type_growth AS
WITH yearly_trend AS (
    SELECT 
        dpt.permit_type,
        dpt.permit_sub_type,
        dd.year,
        COUNT(fp.permit_nbr) AS permit_count
    FROM la_lakehouse.gold.fact_permits AS fp
    LEFT JOIN la_lakehouse.gold.dim_permit_type AS dpt
        ON fp.permit_type_key = dpt.permit_type_key
    LEFT JOIN la_lakehouse.gold.dim_date AS dd
        ON fp.issue_date_key = dd.date_key
    WHERE dd.year BETWEEN 2021 AND 2026
    GROUP BY dpt.permit_type, dpt.permit_sub_type, dd.year
)
SELECT 
    permit_type,
    permit_sub_type,
    year,
    permit_count,
    LAG(permit_count, 1) OVER (
        PARTITION BY permit_type, permit_sub_type 
        ORDER BY year
    ) AS prev_year_count,
    ROUND(
        (permit_count - LAG(permit_count, 1) OVER (PARTITION BY permit_type, permit_sub_type ORDER BY year)) 
        * 100.0 / NULLIF(LAG(permit_count, 1) OVER (PARTITION BY permit_type, permit_sub_type ORDER BY year), 0), 
        2
    ) AS yoy_growth_pct
FROM yearly_trend

ORDER BY permit_type, permit_sub_type, year;

In [0]:
%sql 
-- Which use categories dominate new activity — shifting from single-family toward multi-unit?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_use_category_growth AS
WITH use_cats AS (
    SELECT 
        fp.permit_nbr,
        dd.year,
        CASE 
            WHEN (dut.use_code = 1 AND dut.use_desc = 'Dwelling - Single Family') 
            OR (dut.use_code = 35 AND dut.use_desc = 'Condo-Single Family')
            OR (dut.use_code = 35 AND dut.use_desc = 'Condo-Duplex')
            THEN 'single_family'
            WHEN dut.use_code IN (2, 5) 
            OR (dut.use_code = 35 AND dut.use_desc = 'Condo-Multi Family')
            THEN 'multi_unit'
            WHEN dut.use_desc LIKE '%Accessory Dwelling Unit%' 
            THEN 'adu'
            WHEN dut.use_desc LIKE '%Store%'
            OR dut.use_desc LIKE '%Office%' 
            OR dut.use_desc LIKE '%Restaurant%'
            OR dut.use_desc LIKE '%Shop%'
            THEN 'commercial'
            ELSE 'other' 
        END AS use_category
    FROM la_lakehouse.gold.fact_permits AS fp
    LEFT JOIN la_lakehouse.gold.dim_use_type AS dut
        ON fp.use_type_key = dut.use_type_key
    LEFT JOIN la_lakehouse.gold.dim_date AS dd
        ON fp.issue_date_key = dd.date_key
    WHERE dd.year BETWEEN 2021 AND 2026
),
yearly_counts AS (
    SELECT 
        year,
        use_category,
        COUNT(permit_nbr) AS permit_count
    FROM use_cats
    GROUP BY year, use_category
),
market_share_percentage AS (
    SELECT 
        year,
        use_category,
        permit_count,
        SUM(permit_count) OVER (PARTITION BY year) AS annual_total,
        ROUND(100.0 * permit_count / SUM(permit_count) OVER (PARTITION BY year), 2) AS market_share_pect
    FROM yearly_counts
), 
pivot_summary AS (
    SELECT 
        use_category,
        MAX(CASE WHEN year = 2021 THEN permit_count END) AS count_2021,
        MAX(CASE WHEN year = 2025 THEN permit_count END) AS count_2025,
        MAX(CASE WHEN year = 2021 THEN market_share_pect END) AS share_2021, 
        MAX(CASE WHEN year = 2025 THEN market_share_pect END) AS share_2025
    FROM market_share_percentage
    GROUP BY use_category
)
SELECT
    use_category,
    count_2021,
    count_2025,
    share_2021,
    share_2025,
    ROUND(share_2025 - share_2021, 2) AS net_shift_pct
FROM pivot_summary
ORDER BY net_shift_pct DESC;


#Testing The Views

In [0]:
%sql
SHOW TABLES IN la_lakehouse.gold

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_permit_volume;

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_permit_type_growth;

In [0]:
%sql 
SELECT * 
FROM la_lakehouse.gold.vw_use_category_growth;